# VI) Slot model

## 1) Reference geometry and mesh

In [1]:
# Structural parameters of the machine
pole_pairs = 6
bundles_per_half_slot = 5

# Generate  fine mesh
from utils.geometry import machine_mesh
from ngsolve.webgui import Draw

mesh = machine_mesh(p=pole_pairs, 
                    bundles_per_half_slot=bundles_per_half_slot, 
                    hBundle=0.25e-3,
                    hCorner_stator=0.05e-3,
                    hCorner_shoes=0.05e-3,
                    )
print(f"Generated fine mesh with {mesh.nv} nodes, {mesh.ne} elements")
Draw(mesh)

Generated fine mesh with 15847 nodes, 31612 elements


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

_____
## 2) Reference model

### 2.a) Material properties

In [2]:
# Material properties
mur_iron = 1000
Br_magnets = 1 # remanence in T
sigma_copper = 5.8e7  # S/m

from ngsolve import pi
from utils.physics import magnetization_halbach

mu0 = 4e-7 * pi
nu = mesh.MaterialCF({"core_stator" : 1 / (mu0 * mur_iron)}, default = 1/mu0) 
sigma = mesh.MaterialCF({"slot(.*)_bundle.*" : sigma_copper}, default = 1)
M = mesh.MaterialCF({"rotor" : magnetization_halbach(br = Br_magnets, p = pole_pairs)}) # A/m

### 2.b) Current supply

In [3]:
# Supply parameters
Jrms = 10e6          # A/m²
load_angle = 150     # deg
frequency = 1000     # Hz
winding_type = "distributed" # "concentrated"


from ngsolve import Integrate
from utils.supply import phase_current
from utils.supply import winding_arrangement, bundle_arrangement

Irms = Jrms * Integrate(1, mesh.Materials("slot11_bundle0"))
phase = phase_current(I_rms=Irms, load_angle=load_angle*pi/180)
winding = winding_arrangement(phase = phase, type = winding_type)
bundles_ref = bundle_arrangement(winding = winding, bundles_per_half_slot = bundles_per_half_slot)

### 2.c) Solve reference magneto-harmonic problem

In [4]:
# FE space
curve_order = 1
fem_order = curve_order
dirichlet_bnd = "shaft|out"

from ngsolve import Periodic, H1
fes_ref = Periodic( H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   dirichlet = dirichlet_bnd, 
                   complex = True),  [-1]*7 )

# Resolution
from utils.physics import solve_magnetoharmonic

result_ref = solve_magnetoharmonic(
    fes = fes_ref, 
    frequency = frequency,
    reluctivity = nu,
    magnetization = M,
    conductivity=sigma,
    supply = bundles_ref,
    verbose = 1)

-- START MAGNETOHARMONIC SOLVER --
Solver : pardiso
Setup function space...              done in 478 ms
Assemble matrix...                   done in 2011 ms
Matrix decomposition with pardiso... done in 1453 ms
Assemble right hand side...          done in 325 ms
Solve the problem...                 done in 103 ms
Pack the results...                  total time: 4.374 s
-- END MAGNETOHARMONIC SOLVER --


### 2.d) Display results

In [5]:
# Display the flux lines (isovalues of vector potential z-component)

electric_angle = 0
a_ref = result_ref["solution"]["a"]
from ngsolve import exp
print(f"Magnetic vector potential (T.m) at an electrical angle of {electric_angle:.0f}°")
Draw((a_ref*exp(1j *pi/180 * electric_angle)).real, 
      result_ref["info"]["fes"].mesh,
      settings = {"Objects" : {"Wireframe" : False}})

Magnetic vector potential (T.m) at an electrical angle of 0°


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

BaseWebGuiScene

In [6]:
from utils.physics import current_density
from ngsolve import Norm
from utils.geometry import mask
import numpy as np

J_ref = current_density(result_ref)
Jslot1 = Integrate(J_ref, mesh.Materials("slot1.*"))

mask_bundles =mask(mesh, "slot.*_bundle.*")
print(f"I_slot1 = {Jslot1:.3e} A")
Draw(Norm(J_ref) * mask_bundles, mesh, 
     settings = {"Objects" : {"Wireframe" : False}})

I_slot1 = -2.382e+03+1.375e+03j A


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

BaseWebGuiScene

_____
## 3) Background model

We want to compute the differential magnetic state $a_{diff} = a_{ref} - a_{background}$.
With
- $a_{ref}$ the reference state already computed
- $a_{background}$ a state computed with homogenized current density

### 3.a) Background current supply

In [7]:
bundles_background = bundle_arrangement(winding = winding, 
                             bundles_per_half_slot = bundles_per_half_slot,
                             background = True)

# Display current distribution 
electric_angle = 0 # angle can be changed, try 90 or 180
print(f"Background current (A) at an electric angle of {electric_angle:.0f}°")
Draw((mesh.MaterialCF(bundles_background)*exp(1j*pi/180*electric_angle)).real, mesh,
     settings = {"Objects" : {"Wireframe" : False}})

Background current (A) at an electric angle of 0°


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

BaseWebGuiScene

### 3.b) Solve background problem

In [8]:
result_background = solve_magnetoharmonic(
    fes = fes_ref, 
    frequency = 0, # we don't want any perturbation in the conductors
    reluctivity = nu,
    magnetization = M,
    conductivity=1,
    supply = bundles_background,
    verbose = 1)

-- START MAGNETOHARMONIC SOLVER --
Solver : pardiso
Setup function space...              done in 75 ms
Assemble matrix...                   done in 399 ms
Matrix decomposition with pardiso... done in 323 ms
Assemble right hand side...          done in 243 ms
Solve the problem...                 done in 10 ms
Pack the results...                  total time: 1.054 s
-- END MAGNETOHARMONIC SOLVER --


### 3.c) Display results

In [9]:
# Display the flux lines (isovalues of vector potential z-component)

electric_angle = 0
a_bg = result_background["solution"]["a"]
print(f"Background magnetic vector potential (T.m) at an electrical angle of {electric_angle:.0f}°")
Draw((a_bg*exp(1j *pi/180 * electric_angle)).real, 
      result_background["info"]["fes"].mesh,
      settings = {"Objects" : {"Wireframe" : False}})

Background magnetic vector potential (T.m) at an electrical angle of 0°


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

BaseWebGuiScene

_____
## 4) Differential magnetic state

We plot the difference between background and reference states.
Our goal is to estimate this field inside each slot to evaluate the losses.

In [10]:
electric_angle = 0
a_diff_ref = result_ref["solution"]["a"]  - result_background["solution"]["a"]
print(f"Differential magnetic vector potential (T.m) at an electrical angle of {electric_angle:.0f}°")
Draw((a_diff_ref*exp(1j *pi/180 * electric_angle)).real, 
      result_background["info"]["fes"].mesh,
      settings = {"Objects" : {"Wireframe" : False}})

Differential magnetic vector potential (T.m) at an electrical angle of 0°


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

BaseWebGuiScene

We can check that the differential magnetic field
- is (almost) **orthogonal to air/iron interface**
- is (almost) **orthogonal to the symmetry axis** of the slot

**NB** : our full model is linear, which is a strong assumption.

____
## 5) Slot background model

Our region of interest is limited to a single stator slot. So, the objective is now to simulate only this part of the design!

In [11]:
slot = "slot2"
slot_domain = slot + ".*"
slot_air_domain = slot + ".{1}"

mask_slot = mask(mesh, slot_domain)
mask_slot_air = mask(mesh, slot_air_domain)

print("Regions of interest")
Draw(mask_slot, mesh, settings = {"Objects" : {"Wireframe" : False}})
Draw(mask_slot_air, mesh, settings = {"Objects" : {"Wireframe" : False}})

Regions of interest


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

BaseWebGuiScene

### 5.a) Schur complement

The standard technique to keep only the dofs of interest $x_1$ and eliminate the other $x_2$ from a linear system 

$$ \begin{bmatrix} A & B \\ C & D
\end{bmatrix} \begin{bmatrix} x_1 \\ x_2
\end{bmatrix} 
= \begin{bmatrix} F_1 \\ F_2 \end{bmatrix} $$ 

is to compute its Schur complement. If $D$ is invertible, we can compute $D^{-1}$, multiply the second line by $B D^{-1}$ and substituting $B x_2$ in the first line gives the reduced linear system 

$$ (A - B D^{-1} C) x_1 = F_1 - B D^{-1} F_2 $$


In [12]:
# Select the dofs to keep and eliminate
dofs_slot = result_background["info"]["fes"].GetDofs(mesh.Materials(slot_domain))
freedofs = result_background["info"]["fes"].FreeDofs()
dofs_slot = dofs_slot & freedofs
dofs_ext = (freedofs & freedofs) & ~dofs_slot

# Display
from ngsolve import GridFunction
mask_dof_slot = GridFunction(result_background["info"]["fes"])
mask_dof_slot.vec.FV().NumPy()[dofs_slot] = 1
print("Dofs of interest")
Draw(mask_dof_slot.components[0], mesh, settings = {"Objects" : {"Wireframe" : False}})
mask_dofs_ext = GridFunction(result_background["info"]["fes"])
mask_dofs_ext.vec.FV().NumPy()[dofs_ext] = 1
print("Dofs to condensate")
Draw(mask_dofs_ext.components[0], mesh, settings = {"Objects" : {"Wireframe" : False}})

Dofs of interest


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

Dofs to condensate


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

BaseWebGuiScene

In [13]:
# Compute Schur complement
from utils.linalg import split_mat, split_vec
from scipy.sparse.linalg import spsolve
A, B, C, D = split_mat(result_background["info"]["K"], dofs_slot, dofs_ext)
F1, F2 = split_vec(result_background["info"]["F"], dofs_slot, dofs_ext)

from scipy.sparse.linalg import inv
BinvD = B * inv(D)
schur_complement = A - BinvD*C

KeyboardInterrupt: 

We can check the result of the reduced model is the same than the full model up to floting computation errors:

In [ ]:
reduced_gfu = GridFunction(result_background["info"]["fes"])
reduced_gfu.vec.FV().NumPy()[dofs_slot] = spsolve(schur_complement, F1 - BinvD * F2 )
Draw((Norm(result_background["solution"]["a"] - reduced_gfu.components[0])) * mask_slot, mesh)

NameError: name 'schur_complement' is not defined

### 5.b) Projected boundary conditions
While exact, Schur complement might be long to compute, and lacks flexibility (what if we change the mesh on the boundary?). So, we use a simplifified technique with appropriate boundary conditions to approximate the behavior from outside the slot.

See
> M. Al Eit et al., “Perturbation Finite Element Method for Efficient Copper Losses Calculation in Switched Reluctance Machines To cite this version : Perturbation Finite Element Method for Efficient Copper Losses Calculation in Switched Reluctance Machines,” 2020.

#### i) Neumann projection

##### Naive projection
Project the background trace on the slot boundary using $h\in H(\text{Curl}; \Omega)$

$$\int_{\Omega_s} h^* \cdot h = \int_{\Omega} h^* \cdot \nu \text{Curl}(a)$$

When the mesh gets finer, we have (quite slow!) convergence of the tangential trace.
It is an indicator of discretization error.

In [14]:
from ngsolve import HCurl

fes_ht_curl = HCurl(mesh.Curve(curve_order), 
                   order = fem_order, 
                   definedon = slot_air_domain,
                   complex = True)

from ngsolve import LinearForm, BilinearForm, GridFunction, dx
from utils.physics import Curl

h, h_ = fes_ht_curl.TnT()
K = BilinearForm(h_ * h * dx).Assemble().mat
f = LinearForm(h_ * (nu* Curl(result_background["solution"]["a"])) * dx).Assemble().vec

ht_bg_curl = GridFunction(fes_ht_curl)
ht_bg_curl.vec.data = K.Inverse()*f

##### Consistent projection

See 
> P. Dular, V. Péron, L. Krähenbühl, and C. Geuzaine, “Progressive eddy current modeling via a finite element subproblem method,” vol. 46, pp. 341–348, 2014, doi: 10.3233/JAE-141943.

We use then $h_t = n\times h \in H^1(\Omega)$

$$ \int_{\partial \Omega} a^* h_t = \int_{\Omega} \text{Curl}(a^*) \cdot \nu \text{Curl}(a) $$

The interesting dofs are only the ones on the boundary, so we can assemble only on a single layer and extract a small sub-matrix.

Proceeding this way the projection is consistent (why? to be checked. write the subproblem weak formulation).

In [15]:
dirichlet_slot_bnd  = slot + ".*shoe"
neumann_slot_bnd  = slot + ".*lateral|" +slot + ".*bottom"
robin_slot_bnd = dirichlet_slot_bnd + "|" + neumann_slot_bnd

fes_ht = H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   definedon = slot_air_domain,
                   dirichlet = robin_slot_bnd,
                   complex = True)

from utils.physics import dual_trace
ht_bg_consistent = dual_trace(fes_ht, robin_slot_bnd, nu, a_bg)

#### ii) Reduced slot model

Now that we have our backgroung field and its trace on the boundaries of the slot, we can simulate and obtain the same result in the slot only.

We have the freedom to chose Dirichlet or Neumann boundary conditions; or more generally

$$ \alpha a + (1-\alpha) \nu \text{Curl}(a) \times n = \alpha a_d + (1-\alpha) h_t $$

with $\alpha \in[0,1]$.

First define the slot finite element space

In [16]:
fes_slot = H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   definedon = slot_domain,
                   complex = True)

bundles_bg_slot = bundle_arrangement(winding = winding, 
                             bundles_per_half_slot = bundles_per_half_slot,
                             background = True,
                             only_in= slot_domain)

Then we can solve the slot problem using one projection type or the other.

In [ ]:
choice_projection = "consistent" # naive

eps_dirichlet = 1e-12
alpha = mesh.BoundaryCF({neumann_slot_bnd : 0 , dirichlet_slot_bnd : 1-eps_dirichlet})
from ngsolve import CF, specialcf
t = CF(((0, 1), (-1, 0)), dims=(2, 2)) * specialcf.normal(mesh.Materials(slot_domain))

if choice_projection == "naive":
    result_background_slot = solve_magnetoharmonic(
            fes = fes_slot,             # localized inside the slot
            frequency = 0,
            reluctivity = nu,
            magnetization = M,
            conductivity=sigma,
            supply = bundles_bg_slot,
            # slot boundary conditions
            robin_bnd=robin_slot_bnd,
            robin_coeff=alpha,
            a_dirichlet=a_bg,
            h_tangential= ht_bg_curl.Trace() * t,
            fix1dof=True,
            verbose = 1)
    
elif choice_projection == "consistent":
    result_background_slot = solve_magnetoharmonic(
            fes = fes_slot,             # localized inside the slot
            frequency = 0,
            reluctivity = nu,
            magnetization = M,
            conductivity=sigma,
            supply = bundles_bg_slot,
            # slot boundary conditions
            robin_bnd=robin_slot_bnd,
            robin_coeff=alpha,
            a_dirichlet=a_bg,
            h_tangential= ht_bg_consistent,
            fix1dof=True,
            verbose = 1)

-- START MAGNETOHARMONIC SOLVER --
Solver : superlu
Setup function space...              done in 32 ms
Assemble matrix...                   done in 43 ms
Matrix decomposition with superlu... done in 134 ms
Assemble right hand side...          done in 73 ms
Solve the problem...                 done in 25 ms
Pack the results...                  total time: 0.307 s
-- END MAGNETOHARMONIC SOLVER --


##### Results

In [18]:
a_slot = result_background_slot["solution"]["a"] * mask(fes_slot)
print(f"Slot magnetic vector potential (T.m)")
Draw(a_slot * mask(fes_slot), 
      result_background_slot["info"]["fes"].mesh,
      settings = {"Objects" : {"Wireframe" : False}})

Slot magnetic vector potential (T.m)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

BaseWebGuiScene

In [19]:
a_slot = result_background_slot["solution"]["a"] * mask(fes_slot)
print(f"Relative L2 error: {Integrate(Norm(a_bg - a_slot)**2, mesh.Materials(slot_domain))**0.5 / Integrate(Norm(a_bg)**2, mesh.Materials(slot_domain))**0.5:.3e} T.m")
print(f"Error in magnetic vector potential (T.m)")
Draw(Norm(a_bg - a_slot)/Norm(a_bg), 
      result_background_slot["info"]["fes"].mesh,
      settings = {"Objects" : {"Wireframe" : False}})

Relative L2 error: 1.838e-03 T.m
Error in magnetic vector potential (T.m)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

BaseWebGuiScene

__________
## 6) Approximation of the true field within the slot

Now we want to compute the true field from within the slot model only, using consistent trace exctraction. We study the type of boundary conditions.

In [ ]:
eps_dirichlet = 1e-12
alpha = mesh.BoundaryCF({neumann_slot_bnd : 0 , dirichlet_slot_bnd : 1-eps_dirichlet})



bundles_ref_slot = bundle_arrangement(winding = winding, 
                                      bundles_per_half_slot = bundles_per_half_slot,
                                      only_in= slot_domain)

result_detailed_slot = solve_magnetoharmonic(
    fes = fes_slot,             # localized inside the slot
    frequency = frequency,
    reluctivity = nu,     
    magnetization = M,
    conductivity=sigma,
    supply = bundles_ref_slot,  # true supply
    # slot boundary conditions
    robin_bnd=robin_slot_bnd,
    robin_coeff=alpha,
    a_dirichlet=a_bg,
    #h_tangential= ht_bg_curl.Trace() * t,
    h_tangential= ht_bg_consistent,
    fix1dof=True,
    verbose = 1)


-- START MAGNETOHARMONIC SOLVER --
Solver : pardiso
Setup function space...              done in 114 ms
Assemble matrix...                   done in 201 ms
Matrix decomposition with pardiso... done in 53 ms
Assemble right hand side...          done in 132 ms
Solve the problem...                 done in 4 ms
Pack the results...                  total time: 0.505 s
-- END MAGNETOHARMONIC SOLVER --


In [ ]:
a_dt_slot = result_detailed_slot["solution"]["a"]
norm_a_ref = Integrate(Norm(a_ref)**2, mesh.Materials(slot_domain))**0.5
error = Integrate(Norm(a_dt_slot - a_ref)**2, mesh.Materials(slot_domain))**0.5
print(f"Relative L2 error = {error / norm_a_ref : .2e}")
Draw(Norm((a_dt_slot - a_ref))/Norm(a_ref)*mask_slot, 
mesh, settings = {"Objects" : {"Wireframe" : False}})

Relative L2 error =  6.87e-03


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

BaseWebGuiScene

- Since iron has a very high permeability, the flux lines of the differential/perturbation field are almost orthogonal to the interface, so Neumann boundary condition seems natural: $\alpha = 0$
- However, the air gap interface is not equivalent to magnetic insulation, so $\alpha$ may not be 1 here.

We can define a piecewise constant $\alpha$ at the air gap interface.

In [ ]:
# for loop (dim alpha = 1)

alphaList = 1 - np.logspace(-12, -8, 10)
errorList = []

for al in alphaList:
    alpha = mesh.BoundaryCF({neumann_slot_bnd : 0 , dirichlet_slot_bnd : al})
    result_detailed_slot = solve_magnetoharmonic(
        fes = fes_slot,             # localized inside the slot
        frequency = frequency,
        reluctivity = nu,     
        magnetization = M,
        conductivity=sigma,
        supply = bundles_ref_slot,  # true supply
        # slot boundary conditions
        robin_bnd=robin_slot_bnd,
        robin_coeff=alpha,
        a_dirichlet=a_bg,
        h_tangential= ht_bg_consistent,
        fix1dof=True,
        verbose = 0)
    
    a_dt_slot = result_detailed_slot["solution"]["a"]
    errorList.append(Integrate(Norm(a_dt_slot - a_ref)**2, mesh.Materials(slot_domain))**0.5 / norm_a_ref )
    print(f"1-alpha = {1-al:.2e} | relative L2 error = {errorList[-1]  : .2e}")
    

1-alpha = 1.00e-12 | relative L2 error =  6.87e-03
1-alpha = 2.78e-12 | relative L2 error =  6.86e-03
1-alpha = 7.74e-12 | relative L2 error =  6.86e-03
1-alpha = 2.15e-11 | relative L2 error =  6.86e-03
1-alpha = 5.99e-11 | relative L2 error =  6.85e-03
1-alpha = 1.67e-10 | relative L2 error =  6.83e-03
1-alpha = 4.64e-10 | relative L2 error =  6.81e-03
1-alpha = 1.29e-09 | relative L2 error =  6.89e-03
1-alpha = 3.59e-09 | relative L2 error =  7.67e-03
1-alpha = 1.00e-08 | relative L2 error =  1.11e-02


In [ ]:
# for loop (dim alpha = 2)

alphaList = 1 - np.logspace(-12, -8, 5)
errorList = []

for al1 in alphaList:
    for al2 in alphaList:
        alpha = mesh.BoundaryCF({neumann_slot_bnd : 0 , "slot21.*shoe" : al1, "slot22.*shoe" : al2 })
        result_detailed_slot = solve_magnetoharmonic(
            fes = fes_slot,             # localized inside the slot
            frequency = frequency,
            reluctivity = nu,     
            magnetization = M,
            conductivity=sigma,
            supply = bundles_ref_slot,  # true supply
            # slot boundary conditions
            robin_bnd=robin_slot_bnd,
            robin_coeff=alpha,
            a_dirichlet=a_bg,
            h_tangential= ht_bg_consistent,
            fix1dof=True,
            verbose = 0)
        
        a_dt_slot = result_detailed_slot["solution"]["a"]
        errorList.append(Integrate(Norm(a_dt_slot - a_ref)**2, mesh.Materials(slot_domain))**0.5 / norm_a_ref )
        print(f"1-a1 = {1-al1:.2e} | 1-a2 = {1-al2:.2e} relative L2 error = {errorList[-1]  : .2e}")


1-a1 = 1.00e-12 | 1-a2 = 1.00e-12 relative L2 error =  6.87e-03
1-a1 = 1.00e-12 | 1-a2 = 1.00e-11 relative L2 error =  6.86e-03
1-a1 = 1.00e-12 | 1-a2 = 1.00e-10 relative L2 error =  6.79e-03
1-a1 = 1.00e-12 | 1-a2 = 1.00e-09 relative L2 error =  6.63e-03
1-a1 = 1.00e-12 | 1-a2 = 1.00e-08 relative L2 error =  1.12e-02
1-a1 = 1.00e-11 | 1-a2 = 1.00e-12 relative L2 error =  6.87e-03
1-a1 = 1.00e-11 | 1-a2 = 1.00e-11 relative L2 error =  6.86e-03
1-a1 = 1.00e-11 | 1-a2 = 1.00e-10 relative L2 error =  6.80e-03
1-a1 = 1.00e-11 | 1-a2 = 1.00e-09 relative L2 error =  6.63e-03
1-a1 = 1.00e-11 | 1-a2 = 1.00e-08 relative L2 error =  1.12e-02
1-a1 = 1.00e-10 | 1-a2 = 1.00e-12 relative L2 error =  6.93e-03
1-a1 = 1.00e-10 | 1-a2 = 1.00e-11 relative L2 error =  6.92e-03
1-a1 = 1.00e-10 | 1-a2 = 1.00e-10 relative L2 error =  6.84e-03
1-a1 = 1.00e-10 | 1-a2 = 1.00e-09 relative L2 error =  6.62e-03
1-a1 = 1.00e-10 | 1-a2 = 1.00e-08 relative L2 error =  1.13e-02
1-a1 = 1.00e-09 | 1-a2 = 1.00e-12 relati

We can probably reduce the error further by defining $\alpha$ as a function, and optimize it. To be interesting it should not be longer to compute than the Schur complement.


### Outlook
Can be extended to nonlinear iron material with multi-harmonic model, see

> N. Simpson, D. J. North, S. M. Collins, and P. H. Mellor, “Additive Manufacturing of Shaped Profile Windings for Minimal AC Loss in Electrical Machines,” IEEE Trans. Ind. Appl., vol. 56, no. 3, pp. 2510–2519, 2020, doi: 10.1109/TIA.2020.2975763.